# 07 - AlexNet features + multi-subject Ridge (all 6 subjects)

Same feature source as the organizers' baseline (notebooks 01/02): AlexNet
layer `features.3`, already extracted into `features/consolidated_features.3.npz`
(train) and `features_test/consolidated_features.3.npz` (test).

The only change vs the baseline: instead of training RidgeCV on **one** subject,
we pool the training data of **all six subjects** (sub-01..06) into a single
encoder. AlexNet features are video-level (subject-independent), so a video seen
by k subjects contributes k training rows (same X, different real EEG y). Using
all subjects' TRAINING data is allowed by the challenge rules.

Pipeline (identical recipe to notebook 01, just more training data):
```
AlexNet feature -> StandardScaler -> PCA(0.95) -> StandardScaler -> RidgeCV -> real EEG
```
Then predict test EEG and correlation-match vs the measured test EEG.

- Pure Python 3.11 / numpy / scipy / scikit-learn - no BERG, net2brain, torch,
  or GPU needed (features are precomputed).
- Inputs used exactly as requested: `trn/` (train EEG), `features/` (train AlexNet
  features for the 1000 videos), `tst/` (test EEG), `features_test/` (test AlexNet
  features), `tst_stim/` (ground truth for subjects 01-04).

In [1]:
import numpy as np
from scipy.signal import decimate
import os

# ---------------- Config ----------------
TRAIN_DIR   = './trn'
TEST_DIR    = './tst'
GT_DIR      = './tst_stim'
FEAT_TRN    = './features/consolidated_features.3.npz'
FEAT_TST    = './features_test/consolidated_features.3.npz'

ALL_SUBJECTS = [1, 2, 3, 4, 5, 6]        # pool training over all six
DEV_SUBJECTS = [1, 2, 3, 4]              # have test ground truth for scoring

DEC = 5                                   # 500 Hz -> 100 Hz
tst_ids = list(range(1001, 1103))
N_TEST  = len(tst_ids)

## 1. Load AlexNet features (same keying as notebook 01)
Each video has 8 frame keys `{id:05d}_fNN`; we average them into one vector.

In [2]:
_ztrn = np.load(FEAT_TRN)
_ztst = np.load(FEAT_TST)

def features_for(ids, z):
    rows = []
    for sid in ids:
        keys = [k for k in z.files if k.startswith(f'{int(sid):05d}_')]
        assert keys, f'no features for stimulus {sid}'
        rows.append(np.mean([z[k].flatten() for k in keys], axis=0))
    return np.stack(rows)

# Build a reusable id -> AlexNet feature table for every training video present
_train_ids_present = sorted({int(k.split('_')[0]) for k in _ztrn.files})
_Xtbl = features_for(_train_ids_present, _ztrn)
feat_of = {int(i): _Xtbl[k] for k, i in enumerate(_train_ids_present)}
print('train videos with features:', len(feat_of), '| feature dim:', _Xtbl.shape[1])

# Test features: one vector per test video (shared across subjects)
X_tst = features_for(tst_ids, _ztst)
print('test features:', X_tst.shape)

train videos with features: 968 | feature dim: 139968
test features: (102, 139968)


## 2. Pool (feature, EEG) training pairs across all 6 subjects

In [3]:
X_parts, Y_parts = [], []
n_chan = n_time = None
for s in ALL_SUBJECTS:
    t = np.load(f'{TRAIN_DIR}/sub-{s:02d}_trn.npz')
    ids = np.array(sorted(t.files, key=int), dtype=int)
    eeg = np.stack([t[str(i)] for i in ids])                 # (n, 17, 1500)
    eeg = decimate(eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    n_chan, n_time = eeg.shape[1], eeg.shape[2]
    # Keep only videos that have AlexNet features (all training videos should)
    keep = [j for j, i in enumerate(ids) if int(i) in feat_of]
    ids, eeg = ids[keep], eeg[keep]
    X_parts.append(np.stack([feat_of[int(i)] for i in ids]))
    Y_parts.append(eeg.reshape(len(ids), -1))
    print(f'sub-{s:02d}: {len(ids)} training pairs')

X_trn = np.concatenate(X_parts, axis=0)
Y_trn = np.concatenate(Y_parts, axis=0)
print('pooled training set:', X_trn.shape, '->', Y_trn.shape)

sub-01: 968 training pairs
sub-02: 940 training pairs
sub-03: 931 training pairs
sub-04: 917 training pairs
sub-05: 922 training pairs
sub-06: 821 training pairs
pooled training set: (5499, 139968) -> (5499, 5100)


## 3. Train one shared encoder on the pooled data (same recipe as notebook 01)

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV

sc  = StandardScaler().fit(X_trn)
pca = PCA(n_components=0.95, svd_solver='full').fit(sc.transform(X_trn))
X_trn_p = pca.transform(sc.transform(X_trn))
sc2 = StandardScaler().fit(X_trn_p)
print('PCA components:', X_trn_p.shape[1])

model = RidgeCV(alphas=np.logspace(-3, 6, 20))
model.fit(sc2.transform(X_trn_p), Y_trn)
print('alpha =', model.alpha_)

# Predicted EEG for the 102 test videos (shared across subjects)
X_tst_s = sc2.transform(pca.transform(sc.transform(X_tst)))
pred_tst = model.predict(X_tst_s).reshape(N_TEST, n_chan, n_time)
print('predicted test EEG:', pred_tst.shape)

PCA components: 722
alpha = 12742.749857031322
predicted test EEG: (102, 17, 300)


## 4. Evaluate per subject
For sub-01..04 we score against `tst_stim/`. For sub-05/06 (no ground truth) we
still rank but skip scoring.

In [5]:
def zrows(A):
    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)

P = zrows(pred_tst[:, :, 5:60].reshape(N_TEST, -1))     # predicted, video order
video_ids = np.array(tst_ids)
results = {}

for s in ALL_SUBJECTS:
    tst_eeg = np.load(f'{TEST_DIR}/sub-{s:02d}_tst.npy')
    tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
    gt_path = f'{GT_DIR}/sub-{s:02d}_tst_stim.npy'

    if os.path.exists(gt_path):
        gt = np.load(gt_path)
        order = np.argsort(gt)                           # label-sorted (validation)
        M = zrows(tst_eeg[order][:, :, 5:60].reshape(N_TEST, -1))
        S = M @ P.T / M.shape[1]
        ranked = video_ids[np.argsort(-S, axis=1)]
        truth = gt[order]
        top1 = (ranked[:, 0] == truth).mean()
        top5 = (ranked[:, :5] == truth[:, None]).any(axis=1).mean()
        ranks = (-S).argsort(1).argsort(1)[np.arange(N_TEST), np.arange(N_TEST)] + 1
        results[s] = (top1, top5, np.median(ranks))
        print(f'sub-{s:02d}:  top-1 {top1:.4f}   top-5 {top5:.4f}   median rank {np.median(ranks):.1f}')
    else:
        # no ground truth; rank in ORIGINAL order (no sorting)
        M = zrows(tst_eeg[:, :, 5:60].reshape(N_TEST, -1))
        S = M @ P.T / M.shape[1]
        ranked = video_ids[np.argsort(-S, axis=1)]
        print(f'sub-{s:02d}:  no ground truth - ranked only (shape {ranked.shape})')

if results:
    mt1 = np.mean([results[s][0] for s in results])
    mt5 = np.mean([results[s][1] for s in results])
    print(f'\nMEAN over dev subjects {sorted(results)}:  top-1 {mt1:.4f}   top-5 {mt5:.4f}')

print('\nReference (sub-01, single-subject):')
print('  AlexNet single-subject (nb 02):   top-1 0.0392  top-5 0.1176')
print('  Option B BERG+Ridge (nb 04):      top-1 0.0490  top-5 0.2157')

sub-01:  top-1 0.0392   top-5 0.1176   median rank 23.5
sub-02:  top-1 0.0588   top-5 0.1569   median rank 23.5
sub-03:  top-1 0.0588   top-5 0.1765   median rank 40.0
sub-04:  top-1 0.0294   top-5 0.0980   median rank 48.0
sub-05:  no ground truth - ranked only (shape (102, 102))
sub-06:  no ground truth - ranked only (shape (102, 102))

MEAN over dev subjects [1, 2, 3, 4]:  top-1 0.0466   top-5 0.1373

Reference (sub-01, single-subject):
  AlexNet single-subject (nb 02):   top-1 0.0392  top-5 0.1176
  Option B BERG+Ridge (nb 04):      top-1 0.0490  top-5 0.2157


## Notes
- This isolates ONE change from the baseline: pooling all six subjects' training
  data into a single AlexNet->EEG encoder. Compare its per-subject top-1/top-5 to
  the single-subject AlexNet baseline to see whether more (cross-subject) training
  data helps for this feature.
- Pure Python 3.11 stack (numpy / scipy / scikit-learn); no deep-learning runtime
  needed because AlexNet features are precomputed in features/ and features_test/.
- For a held-out submission with this model, rank sub-05/06 in ORIGINAL order (as
  done in the no-ground-truth branch above) and save with the official filenames.